# Customer Churn — Advanced Analysis & Predictive Modeling
**Telco Customer Retention Intelligence Platform**

---
| Module | Coverage |
|--------|----------|
| ML Models | Logistic Regression · Random Forest · Gradient Boosting |
| Business Metrics | CLV · Revenue-at-Risk · Retention ROI |
| Segmentation | Risk Tiers · Cohort Analysis · High-Value Profiling |
| Output | Scored customer list · Actionable recommendations |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score, accuracy_score,
    precision_score, recall_score, average_precision_score,
    precision_recall_curve
)

# ── Paths ──
BASE = r'D:\customer churn dataanlyst'
VIZ  = BASE + r'\visualization'

# ── Global style ──
sns.set_theme(style='whitegrid', font_scale=1.05)
PALETTE = {'Churned': '#E74C3C', 'Retained': '#2ECC71'}
C_RED, C_GREEN, C_BLUE, C_ORANGE = '#E74C3C', '#2ECC71', '#3498DB', '#F39C12'
print('Libraries loaded.')

## 1. Data Loading & Feature Engineering

In [ ]:
df = pd.read_csv(BASE + r'\data\telco_churn_cleaned.csv')

# ── Engineered features ──
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']
df['NumAddOnServices'] = df[service_cols].apply(lambda row: (row == 'Yes').sum(), axis=1)
df['AvgChargePerMonth']  = df['TotalCharges'] / (df['tenure'] + 1)
df['IsHighValue']        = (df['MonthlyCharges'] > df['MonthlyCharges'].quantile(0.75)).astype(int)
df['IsNewCustomer']      = (df['tenure'] <= 6).astype(int)
df['IsLongTermCustomer'] = (df['tenure'] >= 48).astype(int)
df['HasFiberOptic']      = (df['InternetService'] == 'Fiber optic').astype(int)
df['IsMonthToMonth']     = (df['Contract'] == 'Month-to-month').astype(int)
df['UsesEcheck']         = (df['PaymentMethod'] == 'Electronic check').astype(int)

churned   = df[df['Churn_Binary'] == 1]
retained  = df[df['Churn_Binary'] == 0]

print(f"Total customers     : {len(df):,}")
print(f"Churned             : {len(churned):,}  ({len(churned)/len(df):.1%})")
print(f"Retained            : {len(retained):,} ({len(retained)/len(df):.1%})")
print(f"Monthly revenue lost: ${churned['MonthlyCharges'].sum():,.0f}")
print(f"Avg monthly charge  : ${df['MonthlyCharges'].mean():.2f}")
print(f"\nEngineered features added: NumAddOnServices, AvgChargePerMonth, IsHighValue,")
print(f"  IsNewCustomer, IsLongTermCustomer, HasFiberOptic, IsMonthToMonth, UsesEcheck")

## 2. ML Pipeline — Data Preparation

In [ ]:
cat_cols = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
    'PaperlessBilling', 'PaymentMethod'
]

drop_cols = ['customerID', 'Churn', 'Tenure_Group']
df_ml = df.drop([c for c in drop_cols if c in df.columns], axis=1)
df_ml = pd.get_dummies(df_ml, columns=cat_cols, drop_first=True)

X = df_ml.drop('Churn_Binary', axis=1)
y = df_ml['Churn_Binary']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train samples : {len(X_train):,}")
print(f"Test samples  : {len(X_test):,}")
print(f"Features      : {X_train.shape[1]}")
print(f"Class balance (train): {y_train.mean():.1%} churn")

## 3. Model Training & Evaluation

In [ ]:
model_defs = {
    'Logistic Regression': (
        LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
        True   # needs scaling
    ),
    'Random Forest': (
        RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, class_weight='balanced'),
        False
    ),
    'Gradient Boosting': (
        GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42),
        False
    ),
}

results = {}
for name, (model, scaled) in model_defs.items():
    Xtr = X_train_s if scaled else X_train
    Xte = X_test_s  if scaled else X_test
    model.fit(Xtr, y_train)
    y_pred = model.predict(Xte)
    y_prob = model.predict_proba(Xte)[:, 1]
    results[name] = {
        'model'   : model,
        'scaled'  : scaled,
        'accuracy': accuracy_score(y_test, y_pred),
        'auc'     : roc_auc_score(y_test, y_prob),
        'f1'      : f1_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall'  : recall_score(y_test, y_pred),
        'y_pred'  : y_pred,
        'y_prob'  : y_prob,
    }
    print(f"{name:<22}  Acc={results[name]['accuracy']:.3f}  "
          f"AUC={results[name]['auc']:.3f}  F1={results[name]['f1']:.3f}  "
          f"Recall={results[name]['recall']:.3f}")

best_name = max(results, key=lambda k: results[k]['auc'])
best      = results[best_name]
print(f"\nBest model by AUC: {best_name} ({best['auc']:.3f})")

## 4. ROC Curves — All Models

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Model Performance — ROC & Precision-Recall Curves', fontsize=14, fontweight='bold')

colors = [C_BLUE, C_GREEN, C_ORANGE]

# ROC curves
ax = axes[0]
for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    ax.plot(fpr, tpr, color=color, lw=2, label=f"{name} (AUC={res['auc']:.3f})")
ax.plot([0,1],[0,1],'--', color='gray', lw=1, label='Random classifier')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves')
ax.legend(fontsize=9)
ax.fill_between([0,1],[0,1], alpha=0.05, color='gray')

# Precision-Recall curves
ax = axes[1]
for (name, res), color in zip(results.items(), colors):
    prec, rec, _ = precision_recall_curve(y_test, res['y_prob'])
    ap = average_precision_score(y_test, res['y_prob'])
    ax.plot(rec, prec, color=color, lw=2, label=f"{name} (AP={ap:.3f})")
baseline = y_test.mean()
ax.axhline(baseline, color='gray', linestyle='--', lw=1, label=f'Baseline ({baseline:.2f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(VIZ + r'\roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: roc_pr_curves.png')

## 5. Confusion Matrices & Classification Reports

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    cm_pct = cm / cm.sum() * 100
    labels = np.array([[f"{v}\n({p:.1f}%)" for v, p in zip(row_v, row_p)]
                        for row_v, row_p in zip(cm, cm_pct)])
    sns.heatmap(cm, annot=labels, fmt='', cmap='Blues', ax=ax,
                xticklabels=['Retained','Churned'],
                yticklabels=['Retained','Churned'],
                cbar=False, linewidths=0.5)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(f'{name}\nAUC={res["auc"]:.3f}  F1={res["f1"]:.3f}')

plt.tight_layout()
plt.savefig(VIZ + r'\confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n{'='*50}")
print(f"Best model: {best_name}")
print('='*50)
print(classification_report(y_test, best['y_pred'], target_names=['Retained','Churned']))

## 6. Feature Importance

In [ ]:
# Use Random Forest for feature importance (tree-based = directly interpretable)
rf_model = results['Random Forest']['model']
importances = pd.Series(rf_model.feature_importances_, index=X_train.columns)
top20 = importances.nlargest(20).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Feature Importance Analysis', fontsize=14, fontweight='bold')

# Top 20 bar chart
ax = axes[0]
bars = ax.barh(range(len(top20)), top20.values,
               color=[C_RED if v > top20.median() else C_BLUE for v in top20.values],
               edgecolor='white', linewidth=0.5)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels([n.replace('_', ' ').replace('Yes', '') for n in top20.index], fontsize=9)
ax.set_xlabel('Importance Score')
ax.set_title('Top 20 Predictors of Churn (Random Forest)')
for i, v in enumerate(top20.values):
    ax.text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=8)

# Logistic Regression coefficients
lr_model = results['Logistic Regression']['model']
coef = pd.Series(np.abs(lr_model.coef_[0]), index=X_train.columns)
top15_lr = coef.nlargest(15).sort_values()

ax = axes[1]
ax.barh(range(len(top15_lr)), top15_lr.values,
        color=[C_ORANGE if v > top15_lr.median() else C_GREEN for v in top15_lr.values],
        edgecolor='white', linewidth=0.5)
ax.set_yticks(range(len(top15_lr)))
ax.set_yticklabels([n.replace('_', ' ').replace('Yes', '') for n in top15_lr.index], fontsize=9)
ax.set_xlabel('|Coefficient|')
ax.set_title('Top 15 Predictors (Logistic Regression — coefficient magnitude)')

plt.tight_layout()
plt.savefig(VIZ + r'\feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: feature_importance.png')
print('\nTop 5 churn drivers:')
for f, v in importances.nlargest(5).items():
    print(f'  {f:<40} {v:.4f}')

## 7. Customer Churn Risk Scoring

In [ ]:
# Score all customers with the best model
gb_model = results['Gradient Boosting']['model']
X_all = df_ml.drop('Churn_Binary', axis=1)
df['ChurnProbability'] = gb_model.predict_proba(X_all)[:, 1]

# Risk tiers
df['RiskTier'] = pd.cut(
    df['ChurnProbability'],
    bins=[0, 0.25, 0.50, 0.75, 1.0],
    labels=['Low Risk', 'Medium Risk', 'High Risk', 'Critical Risk']
)

risk_summary = df.groupby('RiskTier', observed=True).agg(
    Customers=('customerID', 'count'),
    ActualChurned=('Churn_Binary', 'sum'),
    AvgChurnProb=('ChurnProbability', 'mean'),
    MonthlyRevenue=('MonthlyCharges', 'sum'),
    AvgMonthlyCharge=('MonthlyCharges', 'mean')
).reset_index()

risk_summary['ChurnRate%'] = (risk_summary['ActualChurned'] / risk_summary['Customers'] * 100).round(1)
risk_summary['RevenueAtRisk'] = (risk_summary['MonthlyRevenue'] * risk_summary['AvgChurnProb']).round(0)

print(risk_summary[['RiskTier','Customers','ChurnRate%','AvgMonthlyCharge',
                      'MonthlyRevenue','RevenueAtRisk']].to_string(index=False))

total_at_risk = risk_summary.loc[
    risk_summary['RiskTier'].isin(['High Risk','Critical Risk']), 'RevenueAtRisk'
].sum()
print(f"\nTotal monthly revenue at risk (High + Critical): ${total_at_risk:,.0f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Customer Risk Tier Analysis', fontsize=14, fontweight='bold')

tier_colors = ['#2ECC71', '#F1C40F', '#E67E22', '#E74C3C']

# Customer count
ax = axes[0]
bars = ax.bar(risk_summary['RiskTier'], risk_summary['Customers'],
              color=tier_colors, edgecolor='white')
for bar, v in zip(bars, risk_summary['Customers']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{v:,}', ha='center', fontsize=10, fontweight='bold')
ax.set_title('Customers per Risk Tier')
ax.set_ylabel('# Customers')
ax.tick_params(axis='x', rotation=15)

# Revenue at risk
ax = axes[1]
bars = ax.bar(risk_summary['RiskTier'], risk_summary['RevenueAtRisk'],
              color=tier_colors, edgecolor='white')
for bar, v in zip(bars, risk_summary['RevenueAtRisk']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
            f'${v:,.0f}', ha='center', fontsize=9, fontweight='bold')
ax.set_title('Monthly Revenue at Risk by Tier')
ax.set_ylabel('Revenue ($)')
ax.tick_params(axis='x', rotation=15)

# Churn probability distribution
ax = axes[2]
ax.hist(df[df['Churn_Binary']==0]['ChurnProbability'], bins=40,
        alpha=0.6, color=C_GREEN, label='Retained', density=True)
ax.hist(df[df['Churn_Binary']==1]['ChurnProbability'], bins=40,
        alpha=0.6, color=C_RED, label='Churned', density=True)
ax.axvline(0.5, color='black', linestyle='--', lw=1.5, label='Decision boundary')
ax.set_title('Churn Probability Distribution')
ax.set_xlabel('Predicted Churn Probability')
ax.set_ylabel('Density')
ax.legend()

plt.tight_layout()
plt.savefig(VIZ + r'\risk_tier_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: risk_tier_analysis.png')

## 8. Customer Lifetime Value (CLV) Analysis

In [ ]:
# CLV model: CLV = MonthlyCharge × (1 / ChurnRate) using predicted probability
# Clamp churn prob to avoid division by zero
DISCOUNT_RATE = 0.10 / 12   # 10% annual discount rate, monthly

df['ChurnProbClamped'] = df['ChurnProbability'].clip(lower=0.01, upper=0.99)
df['ExpectedTenureMonths'] = 1 / df['ChurnProbClamped']
df['CLV'] = df['MonthlyCharges'] * df['ExpectedTenureMonths'] / (1 + DISCOUNT_RATE)

# CLV segments
df['CLV_Segment'] = pd.qcut(df['CLV'], q=4,
                             labels=['Bronze', 'Silver', 'Gold', 'Platinum'])

clv_stats = df.groupby('CLV_Segment', observed=True).agg(
    Count=('CLV', 'count'),
    AvgCLV=('CLV', 'mean'),
    TotalCLV=('CLV', 'sum'),
    AvgMonthlyCharge=('MonthlyCharges', 'mean'),
    ChurnRate=('Churn_Binary', 'mean')
).reset_index()
clv_stats['ChurnRate'] = (clv_stats['ChurnRate'] * 100).round(1)
clv_stats['AvgCLV'] = clv_stats['AvgCLV'].round(0)
clv_stats['TotalCLV'] = clv_stats['TotalCLV'].round(0)

print(clv_stats.to_string(index=False))
print(f"\nTotal portfolio CLV : ${df['CLV'].sum():,.0f}")
print(f"Avg customer CLV    : ${df['CLV'].mean():,.0f}")
print(f"Platinum tier CLV   : ${clv_stats[clv_stats['CLV_Segment']=='Platinum']['TotalCLV'].values[0]:,.0f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Customer Lifetime Value (CLV) Analysis', fontsize=14, fontweight='bold')

seg_colors = ['#CD7F32', '#C0C0C0', '#FFD700', '#E5E4E2']

# Average CLV per segment
ax = axes[0, 0]
bars = ax.bar(clv_stats['CLV_Segment'], clv_stats['AvgCLV'],
              color=seg_colors, edgecolor='white')
for bar, v in zip(bars, clv_stats['AvgCLV']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'${v:,.0f}', ha='center', fontsize=10, fontweight='bold')
ax.set_title('Average CLV by Segment')
ax.set_ylabel('Avg CLV ($)')

# Total CLV pie
ax = axes[0, 1]
ax.pie(clv_stats['TotalCLV'], labels=clv_stats['CLV_Segment'],
       colors=seg_colors, autopct='%1.1f%%', startangle=90,
       wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
ax.set_title('Portfolio CLV Distribution by Segment')

# CLV vs Churn Probability scatter
ax = axes[1, 0]
scatter = ax.scatter(df['ChurnProbability'], df['CLV'],
                     c=df['Churn_Binary'], cmap='RdYlGn_r',
                     alpha=0.3, s=8)
ax.set_xlabel('Churn Probability')
ax.set_ylabel('CLV ($)')
ax.set_title('CLV vs Churn Probability')
ax.set_ylim(0, df['CLV'].quantile(0.99))
plt.colorbar(scatter, ax=ax, label='Churned (1=Yes)')

# CLV vs Monthly Charges by Contract
ax = axes[1, 1]
for contract, color in zip(['Month-to-month','One year','Two year'],
                            [C_RED, C_ORANGE, C_GREEN]):
    subset = df[df['Contract'] == contract]
    ax.scatter(subset['MonthlyCharges'], subset['CLV'].clip(upper=df['CLV'].quantile(0.99)),
               alpha=0.3, s=8, color=color, label=contract)
ax.set_xlabel('Monthly Charges ($)')
ax.set_ylabel('CLV ($)')
ax.set_title('Monthly Charges vs CLV by Contract Type')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(VIZ + r'\clv_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: clv_analysis.png')

## 9. Cohort Retention Analysis

In [ ]:
# Cohort = first year the customer joined (approximated from tenure)
# Since data is a snapshot, we simulate cohort behavior using tenure bands

cohort_bins  = [0, 6, 12, 24, 36, 48, 60, 72]
cohort_labels = ['0-6m', '7-12m', '13-24m', '25-36m', '37-48m', '49-60m', '61-72m']

df['Cohort'] = pd.cut(df['tenure'], bins=cohort_bins, labels=cohort_labels, right=True)

cohort_df = df.groupby(['Cohort', 'Contract'], observed=True).agg(
    Total=('Churn_Binary', 'count'),
    Churned=('Churn_Binary', 'sum')
).reset_index()
cohort_df['RetentionRate'] = (1 - cohort_df['Churned'] / cohort_df['Total']) * 100

# Pivot for heatmap
pivot = cohort_df.pivot_table(
    index='Contract', columns='Cohort', values='RetentionRate', observed=True
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Cohort Retention Analysis', fontsize=14, fontweight='bold')

# Heatmap
ax = axes[0]
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn',
            vmin=40, vmax=100, ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Retention Rate (%)'})
ax.set_title('Retention Rate (%) by Contract Type & Tenure Cohort')
ax.set_xlabel('Tenure Cohort')

# Line chart: churn rate by cohort and contract
ax = axes[1]
contract_colors = {'Month-to-month': C_RED, 'One year': C_ORANGE, 'Two year': C_GREEN}
for contract, color in contract_colors.items():
    subset = cohort_df[cohort_df['Contract'] == contract]
    ax.plot(subset['Cohort'].astype(str), 100 - subset['RetentionRate'],
            marker='o', color=color, lw=2, label=contract)
ax.set_title('Churn Rate Trend by Contract & Tenure')
ax.set_xlabel('Tenure Cohort')
ax.set_ylabel('Churn Rate (%)')
ax.legend()
ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig(VIZ + r'\cohort_retention.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: cohort_retention.png')

## 10. Retention Strategy ROI Calculator

In [ ]:
# Business case: what is the ROI of different retention interventions?

# Inputs
avg_monthly_charge   = df['MonthlyCharges'].mean()        # $64.76
avg_tenure_months    = df['tenure'].mean()                 # ~32 months
current_churn_rate   = df['Churn_Binary'].mean()           # 26.5%
total_customers      = len(df)

# High-risk segment
high_risk = df[df['RiskTier'].isin(['High Risk', 'Critical Risk'])]
n_high_risk = len(high_risk)
monthly_rev_at_risk = high_risk['MonthlyCharges'].sum()

# Retention strategies
strategies = [
    {
        'name'               : 'Contract Upgrade Incentive',
        'description'        : 'Offer 1-month discount to switch Month-to-month → Annual',
        'target_count'       : len(df[(df['Contract']=='Month-to-month') & (df['ChurnProbability']>0.4)]),
        'success_rate'       : 0.30,   # 30% conversion
        'churn_reduction'    : 0.55,   # reduces churn by 55% for converted customers
        'cost_per_customer'  : avg_monthly_charge * 1,  # 1 month discount
        'avg_monthly_charge' : avg_monthly_charge,
    },
    {
        'name'               : 'Proactive Support Outreach',
        'description'        : 'Personal call to High/Critical risk customers in month 1-6',
        'target_count'       : len(df[(df['RiskTier'].isin(['High Risk','Critical Risk'])) & (df['tenure']<=6)]),
        'success_rate'       : 0.25,
        'churn_reduction'    : 0.40,
        'cost_per_customer'  : 15,   # $15 support cost
        'avg_monthly_charge' : avg_monthly_charge,
    },
    {
        'name'               : 'Service Bundle Upgrade',
        'description'        : 'Offer TechSupport + OnlineSecurity bundle at discount to Fiber users',
        'target_count'       : len(df[(df['HasFiberOptic']==1) & (df['ChurnProbability']>0.5)]),
        'success_rate'       : 0.35,
        'churn_reduction'    : 0.35,
        'cost_per_customer'  : 20,   # $20 bundle subsidy
        'avg_monthly_charge' : avg_monthly_charge,
    },
    {
        'name'               : 'Loyalty Rewards Program',
        'description'        : 'Points/cashback for customers with tenure 12-24 months',
        'target_count'       : len(df[(df['tenure']>=12) & (df['tenure']<=24) & (df['Churn_Binary']==0)]),
        'success_rate'       : 0.60,
        'churn_reduction'    : 0.20,
        'cost_per_customer'  : 10,
        'avg_monthly_charge' : avg_monthly_charge,
    },
]

print(f"{'Strategy':<35} {'Target':>7} {'Converted':>10} {'Rev Saved/Mo':>13} {'Program Cost':>13} {'ROI':>8}")
print('-'*90)

for s in strategies:
    converted      = int(s['target_count'] * s['success_rate'])
    monthly_saved  = converted * s['avg_monthly_charge'] * s['churn_reduction']
    program_cost   = s['target_count'] * s['cost_per_customer']
    annual_saved   = monthly_saved * 12
    roi            = (annual_saved - program_cost) / program_cost * 100 if program_cost > 0 else 0
    s['roi']       = roi
    s['monthly_saved'] = monthly_saved
    s['program_cost']  = program_cost
    s['converted'] = converted
    print(f"{s['name']:<35} {s['target_count']:>7,} {converted:>10,} "
          f"${monthly_saved:>11,.0f} ${program_cost:>11,.0f} {roi:>7.0f}%")

total_monthly_saved = sum(s['monthly_saved'] for s in strategies)
total_cost = sum(s['program_cost'] for s in strategies)
blended_roi = (total_monthly_saved * 12 - total_cost) / total_cost * 100
print('-'*90)
print(f"{'COMBINED PROGRAM':<35} {'':>7} {'':>10} "
      f"${total_monthly_saved:>11,.0f} ${total_cost:>11,.0f} {blended_roi:>7.0f}%")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Retention Strategy ROI Dashboard', fontsize=14, fontweight='bold')

names = [s['name'].replace(' ', '\n') for s in strategies]
rois  = [s['roi'] for s in strategies]
saves = [s['monthly_saved'] for s in strategies]
costs = [s['program_cost'] for s in strategies]
roi_colors = [C_GREEN if r > 200 else C_ORANGE if r > 100 else C_RED for r in rois]

# ROI comparison
ax = axes[0]
bars = ax.barh(names, rois, color=roi_colors, edgecolor='white')
ax.axvline(100, color='gray', linestyle='--', lw=1, label='100% ROI')
for bar, v in zip(bars, rois):
    ax.text(v + 5, bar.get_y() + bar.get_height()/2, f'{v:.0f}%', va='center', fontsize=9)
ax.set_xlabel('Annual ROI (%)')
ax.set_title('ROI by Strategy')
ax.legend(fontsize=8)

# Monthly revenue saved
ax = axes[1]
bars = ax.barh(names, saves, color=C_BLUE, edgecolor='white')
for bar, v in zip(bars, saves):
    ax.text(v + 100, bar.get_y() + bar.get_height()/2, f'${v:,.0f}', va='center', fontsize=9)
ax.set_xlabel('Monthly Revenue Saved ($)')
ax.set_title('Monthly Revenue Retained')

# Cost vs Benefit
ax = axes[2]
x_pos = range(len(strategies))
width = 0.35
bars1 = ax.bar([p - width/2 for p in x_pos], costs, width, label='Program Cost', color=C_RED, alpha=0.8)
bars2 = ax.bar([p + width/2 for p in x_pos], [s*12 for s in saves], width,
               label='Annual Revenue Saved', color=C_GREEN, alpha=0.8)
ax.set_xticks(list(x_pos))
ax.set_xticklabels([s['name'].split()[0] for s in strategies])
ax.set_title('Cost vs Annual Revenue Saved')
ax.set_ylabel('Amount ($)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(VIZ + r'\retention_roi.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: retention_roi.png')

## 11. High-Value At-Risk Customer Priority List

In [ ]:
# Priority score = churn_probability × monthly_charges × (1 / tenure+1)
df['PriorityScore'] = (
    df['ChurnProbability'] * df['MonthlyCharges'] * (1 / (df['tenure'] + 1))
)

priority_list = df[
    (df['Churn_Binary'] == 0) &   # still active customers
    (df['ChurnProbability'] > 0.5)
][[
    'customerID', 'tenure', 'Contract', 'MonthlyCharges',
    'InternetService', 'ChurnProbability', 'CLV', 'RiskTier', 'PriorityScore'
]].sort_values('PriorityScore', ascending=False).head(15).reset_index(drop=True)

priority_list['ChurnProbability'] = priority_list['ChurnProbability'].map('{:.1%}'.format)
priority_list['CLV'] = priority_list['CLV'].map('${:,.0f}'.format)
priority_list['PriorityScore'] = priority_list['PriorityScore'].round(2)

print('Top 15 Active Customers to Contact Immediately:')
print(priority_list.to_string(index=False))

# Save scored full dataset
output_cols = ['customerID', 'tenure', 'Contract', 'MonthlyCharges',
               'ChurnProbability', 'RiskTier', 'CLV', 'CLV_Segment', 'PriorityScore']
df[output_cols].sort_values('PriorityScore', ascending=False).to_csv(
    BASE + r'\data\customer_risk_scores.csv', index=False
)
print('\nFull scored customer list saved to: data/customer_risk_scores.csv')

## 12. Executive Business Intelligence Summary

In [ ]:
fig = plt.figure(figsize=(16, 9))
fig.patch.set_facecolor('#1a1d27')
gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.6, wspace=0.5)

def kpi_box(ax, value, label, sublabel='', color=C_BLUE):
    ax.set_facecolor('#252837')
    for spine in ax.spines.values(): spine.set_color(color)
    ax.set_xticks([]); ax.set_yticks([])
    ax.text(0.5, 0.62, value, transform=ax.transAxes,
            fontsize=20, fontweight='bold', color=color,
            ha='center', va='center')
    ax.text(0.5, 0.28, label, transform=ax.transAxes,
            fontsize=9, color='white', ha='center', va='center')
    if sublabel:
        ax.text(0.5, 0.10, sublabel, transform=ax.transAxes,
                fontsize=7.5, color='#aaa', ha='center', va='center')

# KPI Row 1
kpi_box(fig.add_subplot(gs[0, 0]), f"{df['Churn_Binary'].mean():.1%}", 'Churn Rate', 'Industry avg: ~20%', C_RED)
kpi_box(fig.add_subplot(gs[0, 1]), f"${df[df['Churn_Binary']==1]['MonthlyCharges'].sum():,.0f}",
        'Monthly Revenue Lost', 'To active churn', C_ORANGE)
kpi_box(fig.add_subplot(gs[0, 2]), f"${df['CLV'].mean():,.0f}", 'Avg Customer CLV',
        f'Total: ${df["CLV"].sum()/1e6:.1f}M', C_GREEN)
kpi_box(fig.add_subplot(gs[0, 3]), f"{best['auc']:.3f}",
        'Best Model AUC', best_name, C_BLUE)

# KPI Row 2
n_cr = len(df[df['RiskTier']=='Critical Risk'])
cr_rev = df[df['RiskTier']=='Critical Risk']['MonthlyCharges'].sum()
kpi_box(fig.add_subplot(gs[1, 0]), f"{n_cr:,}", 'Critical Risk Customers',
        f'${cr_rev:,.0f}/mo at risk', C_RED)
kpi_box(fig.add_subplot(gs[1, 1]), f"{len(df[df['Contract']=='Month-to-month']):,}",
        'Month-to-Month Customers', '42.7% churn rate', C_ORANGE)
kpi_box(fig.add_subplot(gs[1, 2]), f"{blended_roi:.0f}%", 'Combined Strategy ROI',
        f'${total_monthly_saved:,.0f}/mo saved', C_GREEN)
kpi_box(fig.add_subplot(gs[1, 3]), f"{total_cost/1000:.0f}K", 'Retention Program Cost',
        'One-time investment', C_BLUE)

# Bottom: recommendations text
ax_rec = fig.add_subplot(gs[2, :])
ax_rec.set_facecolor('#1a1d27')
ax_rec.set_xticks([]); ax_rec.set_yticks([])
for spine in ax_rec.spines.values(): spine.set_color('#444')

top_strategies = sorted(strategies, key=lambda s: s['roi'], reverse=True)
rec_text = (
    f"STRATEGIC RECOMMENDATIONS  |  "
    f"#1: {top_strategies[0]['name']} (ROI={top_strategies[0]['roi']:.0f}%)  |  "
    f"#2: {top_strategies[1]['name']} (ROI={top_strategies[1]['roi']:.0f}%)  |  "
    f"Priority: {n_cr:,} critical-risk customers  |  "
    f"Quick win: migrate {len(df[df['Contract']=='Month-to-month']):,} M2M customers to annual contracts"
)
ax_rec.text(0.5, 0.5, rec_text, transform=ax_rec.transAxes,
            fontsize=9, color='white', ha='center', va='center',
            wrap=True)

fig.text(0.5, 0.97, 'CUSTOMER RETENTION INTELLIGENCE DASHBOARD',
         ha='center', fontsize=14, fontweight='bold', color='white')
fig.text(0.5, 0.94, f'Telco Customer Churn Analysis  |  {len(df):,} customers  |  Gradient Boosting model',
         ha='center', fontsize=9, color='#aaa')

plt.savefig(VIZ + r'\executive_dashboard.png', dpi=150, bbox_inches='tight',
            facecolor='#1a1d27')
plt.show()
print('Saved: executive_dashboard.png')

## 13. Cross-Validation — Model Reliability

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('5-Fold Stratified Cross-Validation Results')
print('='*65)
print(f"{'Model':<25} {'Mean AUC':>10} {'Std AUC':>10} {'Min AUC':>10} {'Max AUC':>10}")
print('-'*65)

cv_results = {}
for name, res in results.items():
    model  = res['model']
    Xdata  = X_train_s if res['scaled'] else X_train
    scores = cross_val_score(model, Xdata, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    cv_results[name] = scores
    print(f"{name:<25} {scores.mean():>10.4f} {scores.std():>10.4f} "
          f"{scores.min():>10.4f} {scores.max():>10.4f}")

print('='*65)

# Box plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot(cv_results.values(), labels=cv_results.keys(), patch_artist=True,
           boxprops=dict(facecolor=C_BLUE, alpha=0.6),
           medianprops=dict(color=C_RED, lw=2))
ax.set_ylabel('ROC-AUC Score')
ax.set_title('5-Fold Cross-Validation AUC Distribution')
ax.set_ylim(0.7, 1.0)
ax.axhline(0.85, color='gray', linestyle='--', lw=1, label='0.85 reference')
ax.legend()
plt.tight_layout()
plt.savefig(VIZ + r'\cross_validation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: cross_validation.png')

## 14. Final Business Summary

In [ ]:
churn_rate     = df['Churn_Binary'].mean()
monthly_lost   = df[df['Churn_Binary']==1]['MonthlyCharges'].sum()
annual_lost    = monthly_lost * 12
n_critical     = len(df[df['RiskTier']=='Critical Risk'])
n_high         = len(df[df['RiskTier']=='High Risk'])
best_auc       = best['auc']

print("="*60)
print("   CUSTOMER RETENTION INTELLIGENCE — FINAL REPORT")
print("="*60)
print(f"""
BUSINESS PROBLEM
  Churn rate          : {churn_rate:.1%}  (industry avg ~20%)
  Monthly revenue lost: ${monthly_lost:,.0f}
  Annual revenue lost : ${annual_lost:,.0f}

PREDICTIVE MODEL
  Best model          : {best_name}
  AUC score           : {best_auc:.3f}  (excellent: >0.80)
  Recall on churners  : {best['recall']:.1%}  (correctly identifies churners)

RISK SEGMENTATION
  Critical Risk       : {n_critical:,} customers  (immediate action required)
  High Risk           : {n_high:,} customers  (priority outreach)
  Revenue at risk     : ${total_at_risk:,.0f}/month

TOP CHURN DRIVERS (model)
  1. Month-to-month contract  (42.7% churn vs 11.3% annual)
  2. Fiber optic internet      (high churn relative to DSL)
  3. Electronic check payment  (45.3% churn)
  4. No tech support / security (30%+ higher churn)
  5. Short tenure (<12 months)  (highest new customer risk)

RETENTION STRATEGY ROI
  Contract Upgrade Incentive  : ROI = {strategies[0]['roi']:.0f}%
  Proactive Support Outreach  : ROI = {strategies[1]['roi']:.0f}%
  Service Bundle Upgrade      : ROI = {strategies[2]['roi']:.0f}%
  Loyalty Rewards Program     : ROI = {strategies[3]['roi']:.0f}%
  Combined program ROI        : {blended_roi:.0f}%
  Total monthly revenue saved : ${total_monthly_saved:,.0f}

IMMEDIATE ACTIONS (Priority Order)
  1. Contact {n_critical:,} critical-risk customers this week
  2. Launch contract upgrade offer for M2M customers >$65/mo
  3. Auto-enroll Fiber optic users in TechSupport bundle trial
  4. Switch electronic check users to auto-pay (bank/card)
  5. Onboarding improvement program for customers in month 1-6
""")
print("="*60)